# EDA Dataset: London Indian Takeaway Orders
## Análisis de criterios de evaluación

Este notebook recoge el análisis exploratorio del dataset London Indian 
Takeaway Orders como parte del proceso de selección del dataset para el 
TFG de predicción de demanda en restauración con delivery.

El dataset contiene pedidos reales de dos restaurantes de comida india 
para llevar ubicados en Londres, Reino Unido.

**Fuente:** https://www.kaggle.com/datasets/henslersoftware/19560-indian-takeaway-orders

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style="whitegrid")

# Carga de los dos restaurantes
r1 = pd.read_csv('../data/kaggle_london_indian_takeaway/restaurant-1-orders.csv')
r2 = pd.read_csv('../data/kaggle_london_indian_takeaway/restaurant-2-orders.csv')


In [4]:
print("=== CRITERIO 1: VOLUMEN ===")

# Pedidos únicos por restaurante
r1_pedidos = r1['Order Number'].nunique()
r2_pedidos = r2['Order ID'].nunique()

print(f"Restaurante 1 - Líneas de detalle: {r1.shape[0]}")
print(f"Restaurante 1 - Pedidos únicos: {r1_pedidos}")
print()
print(f"Restaurante 2 - Líneas de detalle: {r2.shape[0]}")
print(f"Restaurante 2 - Pedidos únicos: {r2_pedidos}")
print()
print(f"Total pedidos combinados: {r1_pedidos + r2_pedidos}")

=== CRITERIO 1: VOLUMEN ===
Restaurante 1 - Líneas de detalle: 74818
Restaurante 1 - Pedidos únicos: 13397

Restaurante 2 - Líneas de detalle: 119183
Restaurante 2 - Pedidos únicos: 19658

Total pedidos combinados: 33055


## Criterio 1: Volumen
El dataset contiene más de 33.000 pedidos reales entre los dos 
restaurantes, volumen más que suficiente para entrenar un modelo 
de series temporales.

In [5]:
print("=== CRITERIO 2: GRANULARIDAD TEMPORAL ===")

r1['Order Date'] = pd.to_datetime(r1['Order Date'], dayfirst=True)
r2['Order Date'] = pd.to_datetime(r2['Order Date'], dayfirst=True)

print(f"¿Tiene fecha? Sí")
print(f"¿Tiene hora? Sí")
print()
print(f"Ejemplo Restaurante 1: {r1['Order Date'].iloc[0]}")
print(f"Ejemplo Restaurante 2: {r2['Order Date'].iloc[0]}")

=== CRITERIO 2: GRANULARIDAD TEMPORAL ===
¿Tiene fecha? Sí
¿Tiene hora? Sí

Ejemplo Restaurante 1: 2019-08-03 20:25:00
Ejemplo Restaurante 2: 2019-08-03 21:58:00


## Criterio 2: Granularidad temporal
Ambos restaurantes disponen de fecha y hora exacta de cada pedido,
lo que permite analizar patrones tanto diarios como por franja horaria.

In [6]:
print("=== CRITERIO 3: RANGO TEMPORAL ===")

for i, (df, nombre) in enumerate([(r1, 'Restaurante 1'), (r2, 'Restaurante 2')], 1):
    dias = (df['Order Date'].max() - df['Order Date'].min()).days
    print(f"{nombre}:")
    print(f"  Desde: {df['Order Date'].min()}")
    print(f"  Hasta: {df['Order Date'].max()}")
    print(f"  Días: {dias} ({round(dias/365, 1)} años)")
    print()

=== CRITERIO 3: RANGO TEMPORAL ===
Restaurante 1:
  Desde: 2015-09-01 14:42:00
  Hasta: 2019-08-03 20:25:00
  Días: 1432 (3.9 años)

Restaurante 2:
  Desde: 2015-12-16 19:14:00
  Hasta: 2019-08-03 21:58:00
  Días: 1326 (3.6 años)



## Criterio 3: Rango temporal
Ambos restaurantes cubren aproximadamente 4 años de datos reales,
muy por encima del mínimo de un año necesario para capturar
patrones estacionales completos.

In [7]:
print("=== CRITERIO 4: CALIDAD DE DATOS ===")

for df, nombre in [(r1, 'Restaurante 1'), (r2, 'Restaurante 2')]:
    print(f"{nombre}:")
    print(f"  Valores nulos:\n{df.isnull().sum()}")
    print(f"  Duplicados: {df.duplicated().sum()}")
    print()

=== CRITERIO 4: CALIDAD DE DATOS ===
Restaurante 1:
  Valores nulos:
Order Number      0
Order Date        0
Item Name         0
Quantity          0
Product Price     0
Total products    0
dtype: int64
  Duplicados: 97

Restaurante 2:
  Valores nulos:
Order ID          0
Order Date        0
Item Name         0
Quantity          0
Product Price     0
Total products    0
dtype: int64
  Duplicados: 178



## Criterio 4: Calidad de los datos
- Ninguna de las dos tablas presenta valores nulos.

- Se detectan 97 duplicados en el Restaurante 1 y 178 en el Restaurante 2.
Son pocos en relación al volumen total (menos del 0,2%) y 
se eliminarán en la fase de preprocesado antes de entrenar el modelo.

In [8]:
print("=== CRITERIO 5: ESPECIFICIDAD Y UBICACIÓN ===")
print("Tipo de negocio: Restaurantes de comida india para llevar (takeaway/delivery)")
print("Ubicación documentada: Londres, Reino Unido")
print("Fuente: Kaggle - descripción oficial del dataset")
print()
print("Productos más frecuentes Restaurante 1:")
print(r1['Item Name'].value_counts().head(5))
print()
print("Productos más frecuentes Restaurante 2:")
print(r2['Item Name'].value_counts().head(5))

=== CRITERIO 5: ESPECIFICIDAD Y UBICACIÓN ===
Tipo de negocio: Restaurantes de comida india para llevar (takeaway/delivery)
Ubicación documentada: Londres, Reino Unido
Fuente: Kaggle - descripción oficial del dataset

Productos más frecuentes Restaurante 1:
Item Name
Pilau Rice       4721
Plain Naan       3753
Plain Papadum    3598
Garlic Naan      2628
Onion Bhajee     2402
Name: count, dtype: int64

Productos más frecuentes Restaurante 2:
Item Name
Pilau Rice       8372
Naan             6681
Plain Papadum    6432
Bombay Aloo      4128
Garlic Naan      3921
Name: count, dtype: int64


## Criterio 5: Especificidad de restauración y ubicación
Los datos corresponden a dos restaurantes reales de reparto a domicilio 
ubicados en Londres, Reino Unido, con ubicación documentada 
por el propio autor del dataset.

## Verificación de criterios de evaluación

- ✅ **Volumen:** 33.055 pedidos reales entre los dos restaurantes.
- ✅ **Granularidad temporal:** fecha y hora exacta de cada pedido.
- ✅ **Rango temporal:** casi 4 años de datos (2015-2019).
- ✅ **Calidad de datos:** sin valores nulos, duplicados mínimos (<0,2%).
- ✅ **Especificidad:** restaurantes reales de delivery en Londres, 
  Reino Unido, con ubicación documentada por el autor del dataset.

## Conclusión: dataset seleccionado como alternativa principal

El análisis confirma que este dataset cumple todos los criterios 
de selección establecido.

**Decisión: dataset válido para el desarrollo del proyecto.**